# 1.0 - LOAD PACKAGES

## LOAD PACKAGES AND GET DATA

In [24]:
# 0.1 - DATA PREP AND MATH LIBRARIES
import pandas       as pd
import numpy        as np
import re

# PLOTTING
import seaborn      as sns
from   matplotlib                 import pyplot as plt
#%matplotlib inline 


# DECISION TREES
from   sklearn.feature_extraction import DictVectorizer        # CONVERT DATAFRAME TO MATRIX AND APPLY ONE-HOT ENCODING
from   sklearn.model_selection    import train_test_split      # SPLIT THE DATASET
from   sklearn.metrics            import (
                                    roc_auc_score,              # CLASSIFICATION ACCURACY METRIC
                                    mean_squared_error         # REGRESSION ACCURACY METRIC
                                )
from   sklearn.tree               import (
                                    DecisionTreeClassifier,     # CLASSIFICATION PREDICTIONS 
                                    DecisionTreeRegressor,      # REGRESSION PREDICTIONS
                                    export_text                # 
                                )
from   sklearn.ensemble           import RandomForestRegressor # ENEMBLE LEANING

###########################################################################################
## GET DATA
###########################################################################################
data_path  = "C:/Users/duffl/anaconda3/_CT161_GitHub_Repos/ml-zoomcamp-homework/07_midterm_project_plants/data/Australian_plant_translocation_database_PUBLIC_2018(ALL_TRANSLOCATIONS).csv"
df         = pd.read_csv(data_path)


# 2.0 - HELPER FUNCTIONS

In [19]:
def fn_clean_text(value):
    if not isinstance(value, str):
        return value

    value = value.lower()
    value = value.replace('%', '_pct_')
    value = re.sub(r'[\s()/\-]+', '_', value)  # spaces, slashes, parentheses, hyphens
    value = re.sub(r"""['"]""", '', value)     # remove quotes
    value = re.sub(r'[,;.?]', '', value)       # remove punctuation
    return value
    
def fn_col_value_status_table(pandas_dataframe):
    print("===================================================================================================================")
    print(f"  VALUES\tMISSING\t\t%_MISSING\tCOLUMN WITHIN THE PANDAS DATAFRAME")
    print("===================================================================================================================")
    
    for column_name in df.columns:
        non_missing     = pandas_dataframe[column_name].count()
        missing         = pandas_dataframe[column_name].isna().sum()
        total           = non_missing + missing
        percent_missing = (missing / total * 100) if total else 0
    
        print(f"  {non_missing}\t\t{missing}\t\t{percent_missing:.2f}%\t\t{column_name}")
    print("===================================================================================================================")


# 3.0 - BEGIN PREPARING DATA

In [25]:
###########################################################################################
#   3.1 - CLEAN COLUMN NAMES AND RECORD VALUES
###########################################################################################
df.columns             = [fn_clean_text(column) for column in df.columns]
df                     = df.map(fn_clean_text)

# PREVIEW NEW  COLUMN NAMES 
fn_col_value_status_table(df)

  VALUES	MISSING		%_MISSING	COLUMN WITHIN THE PANDAS DATAFRAME
  1182		0		0.00%		status_summary
  1182		0		0.00%		translocation_number
  1182		0		0.00%		treatment_number
  321		861		72.84%		treatment_description
  1182		0		0.00%		species
  1182		0		0.00%		family
  1182		0		0.00%		lifeform
  1182		0		0.00%		state
  1166		16		1.35%		conservation_status_epbc_
  1156		26		2.20%		conservation_status_state_
  1182		0		0.00%		latitude_decimal_degrees_
  1182		0		0.00%		longitude_decimal_degrees_
  1182		0		0.00%		bioregion
  1182		0		0.00%		broad_habitat_group
  1166		16		1.35%		habitat_description
  871		311		26.31%		research_to_support_translocation_
  1177		5		0.42%		mitigation_translocation
  1110		72		6.09%		habitat_condition_eg_highly_disturbed_small_remnant_large_remnant_
  1182		0		0.00%		translocation_type
  775		407		34.43%		start_month
  1138		44		3.72%		start_year
  892		290		24.53%		number_of_subsequent_plantings
  91		1091		92.30%		number_of_seeds_direct_seeding_only_
  1054		12

In [ ]:
# PREVIEW DATA VALUES 
df.head(20)

In [26]:
###########################################################################################
#   3.2 - DROP RECORDS DATASET CREATORS HAVE MARKED AS UNSUITABLE FOR ANALYSIS
###########################################################################################
exclude_status_summary = r'incomplete|unable_to_monitor|finalised '

df                     = df.loc[
                          ~df['status_summary'].str.contains(
                            exclude_status_summary, 
                            na=False
                          )
                        ].copy()

In [27]:
###########################################################################################
#   3.3 - DROP FEATURES WHOSE CONTENT IS UNIQUE ACROSS ALL RECORDS OR OF LOW FREQUENCY 
###########################################################################################
exclude_features_list  =    [
                                'status_summary',
                                'treatment_description', 
                                'species', 
                                'family',
                                'bioregion',
                                'conservation_status_state_',
                                'latitude_decimal_degrees_',
                                'longitude_decimal_degrees_',
                                'habitat_description',
                                'start_month',
                                'start_year',
                                'number_of_seeds_direct_seeding_only_',
                                'number_of_individuals_planted_or_germinated_if_direct_seeding_total_',
                                'number_planted_notes',
                                'watering_in_only', 
                                'watering_<6_months_', 
                                'watering_>6_months_or_in_dry_season_', 
                                'last_monitoring_date',
                                'number_of_plants_surviving',
                                'number_of_subsequent_plantings',
                                'no_second_generation_plants',
                                'short_term_success__pct__of_plants_which_survived_first_year_', 
                                'medium_term_success_population_persisting_at_secure_site_',
                                'long_term_success_self_sustaining_population_established_with_dynamics_comparable_to_natural_populations_',
                                'reasons_for_failure_or_sub_optimal_performance_or_success_if_known_',
                                'failure_factors_summary',
                                'success_factors_summary',
                                'notes',
                                'project_managed_by',
                                'contact_person_s', 
                                'references' 
                            ]

missing                = [col for col in exclude_features_list if col not in df.columns]
print("Features to remove that were not found:\r\n", missing)
print()

# DROP COLUMNS IN LIST
df                     = df.drop(columns=exclude_features_list, errors='ignore')

# DEBUG SHOW NEW COLUMN NAMES
print("===================================================================================================================")
print("===== REMAINING FEATURES ==========================================================================================")
fn_col_value_status_table(df)


Features to remove that were not found:
 []

===== REMAINING FEATURES ==========================================================================================
  VALUES	MISSING		%_MISSING	COLUMN WITHIN THE PANDAS DATAFRAME
  841		0		0.00%		translocation_number
  841		0		0.00%		treatment_number
  841		0		0.00%		lifeform
  841		0		0.00%		state
  830		11		1.31%		conservation_status_epbc_
  841		0		0.00%		broad_habitat_group
  665		176		20.93%		research_to_support_translocation_
  841		0		0.00%		mitigation_translocation
  838		3		0.36%		habitat_condition_eg_highly_disturbed_small_remnant_large_remnant_
  841		0		0.00%		translocation_type
  806		35		4.16%		propagule_type_seed_seedling_cutting_tissue_culture_whole_plants_etc_
  692		149		17.72%		soil_preparation_y_n_
  692		149		17.72%		pre_planting_burn_y_n_
  681		160		19.02%		pre_planting_slashing_or_weeding_y_n_
  690		151		17.95%		pre_site_preparation_all_
  684		157		18.67%		herbivore_exclosure_fence_y_n_
  684		157		18.67%		herbivore

In [30]:
##########################################################################################
# NORMALIZE PERCENTAGE OF PLANTS THAT SURVIVED TRANSPLANTING
###########################################################################################
source = df['plants_surviving__pct__'].astype('string').str.strip()

replacements = {
    '<NA>'                          : '0.00',
    '<na>'                          : '0.00',
    '??'                            : '0.00',
    '#div/0!'                       : '0.00',
    '#ref!'                         : '0.00',
    '#value!'                       : '0.00',
    'apr_00'                        : '0.00',
    'n_a_annual_'                   : '0.00',
    'n_a_for_annual_species'        : '0.00',
    'n_a_annual_plant_'             : '0.00',
    'n_a_direct_seeding_'           : '0.00',
    'n_a_short_lived_'              : '0.00',
    'Not monitored yet'             : '0.00',
    'unknown_probably_0_'           : '0.00',
    '0_but_substantial_recruitment_': '0.00',
    '<5'                            : '5.00',
    '<10'                           : '10.00',
    '>10'                           : '10.00',
    '>10_pct_'                      : '10.00',
    '16_pct_'                       : '16.00',
    '4_pct_'                        : '4.00', 
    '11_pct_'                       : '11.00',
    'c20'                           : '20.00',
    'c25_pct_'                      : '25.00',
    '17_pct_'                       : '17.00',
    '<20'                           : '20.00',
    'c.20'                          : '20.00',
    '>25'                           : '25.00',
    '>25_pct_'                      : '25.00',
    'c.25'                          : '25.00',
    '20-40_pct_'                    : '30.00',
    '20-40'                         : '30.00',
    'c.25'                          : '25.00',
    '29_pct_'                       : '29.00',
    '~30'                           : '30.00',
    '>30'                           : '30.00',
    '>33'                           : '33.00',
    '~40'                           : '40.00',
    '>40'                           : '40.00',
    '<50?'                          : '50.00',
    '<50_pct_'                      : '50.00',
    '<50_pct_?'                     : '50.00',
    '<50?'                          : '50.00',
    '50_pct_'                       : '50.00',
    '~50'                           : '50.00',
    '>50_pct_'                      : '50.00',
    '>50'                           : '50.00',
    '>50_pct_'                      : '50.00',
    '52_pct_'                       : '52.00',
    '60_pct_'                       : '60.00',
    '69_pct_'                       : '69.00',
    'unknown_but_probably_>75_pct_' : '75.00',
    '77_pct_'                       : '77.00',
    '80_pct_'                       : '80.0',
    '80_pct__across_all_sites_and_treatments_'  : '80.00',
    '80_pct_ (across all sites and treatments)' : '80.00',
    '83_pct_'                       : '83.00',
    '85_pct_'                       : '85.0',
    '>90'                           : '90.00',
    '>95'                           : '95.00',
    '>95_pct_'                      : '95.00',
    '100_pct_'                      : '100.00',
    '>100_with_recruitment_'        : '0.00',
    'good'                          : '0.00',
    'unknown'                       : '0.00',
    'Unknown'                       : '0.00',
    'unknown_number_germinating_not_documented_' : '0.00',
    'unknown_how_many_were_planted_in_total'     : '0.00',
    'unknown_number_of_germinants_not_counted_'  : '0.00',
    'too_early'                     : '0.00',
    'plants_in_the_ground_for_<_12_months'       : '0.00',
    '<12_months_since_planting'     : '0.00',
    '<_12_mnths_since_translocation': '0.00',
    '<12_months_elapsed'            : '0.00',
    '_pct_'                         : '0.00',
}

cleaned = (
    source.replace(replacements)
          .str.replace('%', '', regex=False)
          .str.strip()
)

parts = cleaned.str.split(':',n=1)
print(df['plants_surviving__pct__'], parts)

# CREATE NEW COLUMN
df['col_plants_surviving_cnt'] = (
    pd.to_numeric(parts.str[0], errors='coerce')
      .fillna(0.0)
)

df['col_plants_surviving_pct'] = (
    pd.to_numeric(parts.str[1], errors='coerce')
      .fillna(0.0)
)
# df 'col_plants_surviving_pct'] = parts.str[0].astype('string').str.strip()
# df['col_plants_surviving_ct'] = parts.str[1].astype('string').str.strip()

print("===== 'col_plants_surviving_pct' cleaned ===============")


# print(f"===== {cleaned[values.isna()].value_counts(dropna=False)} ===============")


print(f"\r\n\r\ndf['col_plants_surviving_cnt'].unique\t\t{df['col_plants_surviving_cnt'].unique}")
print(f"\r\n\r\ndf['col_plants_surviving_pct'].unique\t\t{df['col_plants_surviving_pct'].unique}")

2                      NaN
3                      NaN
4                      NaN
5                     6200
7                      000
               ...        
1169                  8482
1170                  8700
1171                 10000
1173    <12_months_elapsed
1177                  5900
Name: plants_surviving__pct__, Length: 841, dtype: object 2          <NA>
3          <NA>
4          <NA>
5        [6200]
7         [000]
         ...   
1169     [8482]
1170     [8700]
1171    [10000]
1173     [0.00]
1177     [5900]
Name: plants_surviving__pct__, Length: 841, dtype: object
===== 'col_plants_surviving_pct' cleaned ===============


df['col_plants_surviving_cnt'].unique		<bound method Series.unique of 2           0.0
3           0.0
4           0.0
5        6200.0
7           0.0
         ...   
1169     8482.0
1170     8700.0
1171    10000.0
1173        0.0
1177     5900.0
Name: col_plants_surviving_cnt, Length: 841, dtype: float64>


df['col_plants_surviving_pct'].unique		<bou

In [31]:
# DEBUG 
col_list = [
    'col_plants_surviving_pct'
]
for col in col_list:
    print(f"{col} ({df[col].nunique()} unique values):")
    for value, count in df[col].value_counts(dropna=False).items():
        print(f"    {value}: {count}")

col_plants_surviving_pct (1 unique values):
    0.0: 841


In [14]:
df[['col_plants_surviving_cnt','col_plants_surviving_pct']]

,col_plants_surviving_cnt,col_plants_surviving_pct
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
5,6200.0,0.0
7,0.0,0.0
...,...,...
1169,8482.0,0.0
1170,8700.0,0.0
1171,10000.0,0.0
1173,0.0,0.0


In [ ]:
df.col_plants_surviving_pct.head()